# Our approach — Gemma 4 31B on COM2SENSE (497 claims, earlier sample from the training split)

Facts written by the model, PRO/CON arguments, relation audit, bipolar framework, extensions and decision rules, run in batches of 100. Results: credulous 82.1, skeptical 82.3, net support 85.3, 3.6% of claims with more than one extension.

Run on Kaggle; the API key is read from Kaggle secrets (`OPENROUTER_API_KEY`). Outputs are kept so the reported numbers can be checked.

In [6]:
!pip install -q openai datasets scikit-learn

In [7]:
import os, re, json, time, pickle
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/gemma_com2sense"); WORK_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"),
                base_url="https://openrouter.ai/api/v1")
MODEL_NAME = "google/gemma-4-31b-it:free"

ds = load_dataset("tasksource/com2sense", split="train")
df = ds.to_pandas()
df["gold"] = df["label"].apply(lambda x: 1 if str(x).strip().lower()=="true" else 0)
df["claim"] = df["sent"].astype(str)
true_ex = df[df["gold"]==1].sample(250, random_state=42)
false_ex = df[df["gold"]==0].sample(250, random_state=42)
baf_df = pd.concat([true_ex, false_ex]).sample(frac=1, random_state=42).reset_index(drop=True)
baf_df["example_index"] = range(len(baf_df))
print("COM2SENSE loaded:", len(baf_df), "| first:", baf_df.iloc[0]["claim"][:50])

COM2SENSE loaded: 500 | first: Tim was happy that November was coming up since he


In [8]:
BAF_CONFIG = {
    "n_pro_candidates": 5, "n_con_candidates": 5, "max_pro_keep": 4, "max_con_keep": 4,
    "min_relevance": 0.60, "min_grounding": 0.55, "min_clarity": 0.60, "min_quality": 0.58,
    "min_target_match": 0.65, "min_grounding_edge": 0.55, "min_relation_score": 0.68,
    "max_conflict_pairs": 3, "max_claim_attacks": 2, "audit_batch_size": 16,
    "generation_temperature": 0.15, "audit_temperature": 0.0, "max_retries": 5,
}
def clip01(v):
    try: return max(0.0,min(1.0,float(v)))
    except: return 0.0
def extract_json_object(text):
    text=str(text or "").strip(); text=re.sub(r"```(?:json)?","",text,flags=re.IGNORECASE).strip()
    try: return json.loads(text)
    except json.JSONDecodeError: pass
    s=text.find("{"); e=text.rfind("}")
    if s>=0 and e>s: return json.loads(text[s:e+1])
    raise ValueError("No JSON")
def deepseek_json_call(prompt, temperature, max_tokens, retries=5):
    last=None
    for a in range(1,retries+1):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You are an expert in claim verification, formal argumentation and strict JSON output."},
                          {"role":"user","content":prompt}],
                temperature=temperature,max_tokens=max_tokens)
            t=resp.choices[0].message.content
            if not t: raise ValueError("empty")
            return extract_json_object(t)
        except Exception as e:
            last=e
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(12); continue
            if a<retries: time.sleep(3*a)
    raise RuntimeError(f"failed: {last}")

def generate_facts(claim):
    prompt = f"""You are building atomic factual building blocks, in the style of a reasoning dataset.

For the claim below, produce 3-5 SHORT, SEPARATE factual statements. Each must be:
- A single atomic fact about ONE concept in the claim.
- General commonsense knowledge, stated on its own.
- NOT a comparison, ranking, or conclusion.
- NOT a statement about whether the claim is true or false.

Claim: "{claim}"

Return 3-5 atomic facts, one per line. Do not conclude."""
    for _ in range(5):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You output atomic factual building blocks. You never conclude."},
                          {"role":"user","content":prompt}], temperature=0.3, max_tokens=400)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(12); continue
            return ""
    return ""
print("✅ CELL 3 ready (helpers + facts)")

✅ CELL 3 ready (helpers + facts)


In [9]:
def generate_and_score_arguments(claim, facts_text, config=BAF_CONFIG):
    prompt = f"""
Construct candidate arguments for a Bipolar Argumentation Framework used for claim verification.

CLAIM:
{claim}

AVAILABLE FACTS:
{facts_text}

First determine the intended meaning of the claim, then reason carefully from facts and commonsense.

Generate up to {config["n_pro_candidates"]} PRO candidates and up to {config["n_con_candidates"]} CON candidates.

PRO: a specific, grounded reason the claim is TRUE.
CON: a specific, grounded reason the claim is FALSE.

CRITICAL rules for CON arguments (attacks):
- Only generate a CON if there is a GENUINE, SPECIFIC, STRONG reason the claim is false.
- Do NOT generate speculative CONs like "maybe not always", "it might not", "in some cases".
- Do NOT generate a CON that merely raises a weak possibility; it must give a real reason the claim is false.
- If the claim is clearly true and there is no strong objection, generate ZERO CON arguments.

CRITICAL rules for PRO arguments:
- Only generate a PRO if there is genuine support. If the claim is clearly false, generate ZERO PRO arguments.
- Do NOT generate circular or vague arguments.

General:
- Each argument must be specific and grounded, under 30 words.
- Do not output the final TRUE/FALSE label.

Score each candidate 0 to 1 for: relevance; grounding; clarity; overall_quality.
Score speculative, weak, vague, or circular arguments BELOW 0.4 on overall_quality.

Return valid JSON only:
{{
  "interpreted_claim": "brief meaning",
  "pro": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}],
  "con": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}]
}}
""".strip()
    raw=deepseek_json_call(prompt,config["generation_temperature"],1900,config["max_retries"])
    out={"interpreted_claim":str(raw.get("interpreted_claim","")).strip(),"pro":[],"con":[]}
    for role in ("pro","con"):
        for e in (raw.get(role,[]) if isinstance(raw.get(role,[]),list) else []):
            if not isinstance(e,dict): continue
            t=str(e.get("text","")).strip()
            if not t: continue
            out[role].append({"text":t,"relevance":clip01(e.get("relevance")),"grounding":clip01(e.get("grounding")),"clarity":clip01(e.get("clarity")),"overall_quality":clip01(e.get("overall_quality"))})
    return out

def select_arguments(gen, config=BAF_CONFIG):
    nodes={"CLAIM":{"role":"CLAIM","text":""}}; scored=[]
    for src,role,pre,mx in [("pro","PRO","P",config["max_pro_keep"]),("con","CON","N",config["max_con_keep"])]:
        cands=[]
        for it in gen.get(src,[]):
            rl,g=clip01(it.get("relevance")),clip01(it.get("grounding")); cl,q=clip01(it.get("clarity")),clip01(it.get("overall_quality"))
            rank=0.30*rl+0.30*g+0.15*cl+0.25*q; sc={**it,"role":role,"rank_score":rank}; scored.append(sc)
            if rl>=config["min_relevance"] and g>=config["min_grounding"] and cl>=config["min_clarity"] and q>=config["min_quality"]: cands.append(sc)
        cands.sort(key=lambda x:x["rank_score"],reverse=True)
        for n,it in enumerate(cands[:mx],1):
            nodes[f"{pre}{n}"]={"role":role,"text":it["text"],"relevance":it["relevance"],"grounding":it["grounding"],"clarity":it["clarity"],"overall_quality":it["overall_quality"],"rank_score":it["rank_score"]}
    return {"nodes":nodes,"all_scored":scored}

def build_relation_candidates(nodes):
    pro=[n for n,i in nodes.items() if i["role"]=="PRO"]; con=[n for n,i in nodes.items() if i["role"]=="CON"]; c=[]
    for p in pro: c.append({"source":p,"target":"CLAIM","allowed":["SUPPORT","NONE"]})
    for n in con: c.append({"source":n,"target":"CLAIM","allowed":["ATTACK","NONE"]})
    for p in pro:
        for n in con:
            c.append({"source":p,"target":n,"allowed":["ATTACK","NONE"]}); c.append({"source":n,"target":p,"allowed":["ATTACK","NONE"]})
    return c
print("✅ piece 1 ready")

✅ piece 1 ready


In [10]:
def audit_baf_relations(claim, facts_text, nodes, config=BAF_CONFIG, batch_size=16):
    cands=build_relation_candidates(nodes)
    at="\n".join(f"{nid} [{i['role']}]: {i['text']}" for nid,i in nodes.items() if nid!="CLAIM")
    aud=[]
    for bs in range(0,len(cands),batch_size):
        b=cands[bs:bs+batch_size]
        pb="\n".join(f"{it['source']} -> {it['target']} | allowed={it['allowed']}" for it in b)
        prompt=f"""Audit candidate relations for a Bipolar Argumentation Framework.

CLAIM:
{claim}
FACTS:
{facts_text}
ARGUMENTS:
{at}

For every candidate, choose exactly one allowed type.
SUPPORT: source gives a meaningful direct reason to accept target.
ATTACK: source directly contradicts, rebuts or undermines target.
NONE: no meaningful direct relation.
Use ATTACK for CON->CLAIM only if the CON gives a genuine reason the claim is false or doubtful.
Use NONE when the CON concedes the claim or agrees.
Keep every reason at eight words or fewer.
For ATTACK/SUPPORT scores normally exceed 0.50; for NONE normally below 0.50.

CANDIDATES:
{pb}

Return valid JSON only:
{{"relations":[{{"source":"P1","target":"CLAIM","type":"SUPPORT","target_match":0.9,"grounding":0.9,"relation_score":0.9,"reason":"..."}}]}}"""
        raw=deepseek_json_call(prompt,config["audit_temperature"],2200,config["max_retries"])
        exp={(it["source"],it["target"]) for it in b}
        allow={(it["source"],it["target"]):set(it["allowed"]) for it in b}
        ret={}
        for r in (raw.get("relations",[]) if isinstance(raw.get("relations",[]),list) else []):
            if not isinstance(r,dict): continue
            s,t=str(r.get("source","")).strip(),str(r.get("target","")).strip()
            typ=str(r.get("type","NONE")).strip().upper(); pr=(s,t)
            if pr not in exp or typ not in allow[pr]: continue
            ret[pr]={"source":s,"target":t,"type":typ,"target_match":clip01(r.get("target_match")),"grounding":clip01(r.get("grounding")),"relation_score":clip01(r.get("relation_score")),"reason":str(r.get("reason","")).strip()}
        for it in b:
            pr=(it["source"],it["target"])
            if pr not in ret:
                ret[pr]={"source":pr[0],"target":pr[1],"type":"NONE","target_match":0.0,"grounding":0.0,"relation_score":0.0,"reason":"omitted"}
        aud.extend(ret.values())
    return {"all_audited_relations":aud}
print("✅ piece 2 ready")

✅ piece 2 ready


In [11]:
def relation_passes(r,config=BAF_CONFIG): return r["target_match"]>=config["min_target_match"] and r["grounding"]>=config["min_grounding_edge"] and r["relation_score"]>=config["min_relation_score"]
def relation_lookup(rels): return {(r["source"],r["target"]):r for r in rels}
def select_formal_supports(rels,config=BAF_CONFIG): return sorted(set((r["source"],r["target"]) for r in rels if r["type"]=="SUPPORT" and r["target"]=="CLAIM" and r["source"].startswith("P") and relation_passes(r,config)))
def select_con_claim_attacks(rels,config=BAF_CONFIG):
    p=[r for r in rels if r["type"]=="ATTACK" and r["target"]=="CLAIM" and r["source"].startswith("N") and relation_passes(r,config)]
    p.sort(key=lambda x:x["relation_score"],reverse=True); return p[:config["max_claim_attacks"]]
def select_independent_conflict_pairs(rels,config=BAF_CONFIG):
    lk=relation_lookup(rels); cand=[]
    pro=sorted({r["source"] for r in rels if r["source"].startswith("P")}); con=sorted({r["source"] for r in rels if r["source"].startswith("N")})
    for p in pro:
        for n in con:
            pc,cp=lk.get((p,n)),lk.get((n,p))
            if not pc or not cp: continue
            if pc["type"]=="ATTACK" and cp["type"]=="ATTACK" and relation_passes(pc,config) and relation_passes(cp,config):
                cand.append({"pro":p,"con":n,"pair_score":min(pc["relation_score"],cp["relation_score"])})
    cand.sort(key=lambda x:x["pair_score"],reverse=True); sel=[]; up=set(); uc=set()
    for c in cand:
        if c["pro"] in up or c["con"] in uc: continue
        sel.append(c); up.add(c["pro"]); uc.add(c["con"])
        if len(sel)>=config["max_conflict_pairs"]: break
    return sel,cand
def support_closure(nodes,sup):
    cl=set(sup); ch=True
    while ch:
        ch=False; new=set()
        for a,b in cl:
            for c,d in cl:
                if b==c and (a,d) not in cl: new.add((a,d))
        if new: cl|=new; ch=True
    return cl
def derive_attacks(nodes,direct,sup):
    sc=support_closure(nodes,sup); sa=set(); se=set()
    for s,sd in sc:
        for at,t in direct:
            if sd==at: sa.add((s,t))
    for at,ad in direct:
        for s,sd in sc:
            if ad==s: se.add((at,sd))
    eff=(set(direct)|sa|se); eff={e for e in eff if e[0]!=e[1]}
    return {"support_closure":sorted(sc),"supported_attacks":sorted(sa),"secondary_attacks":sorted(se),"effective_attacks":sorted(eff)}
print("✅ piece 3 ready")

✅ piece 3 ready


In [12]:
def all_subsets(items):
    il=sorted(items)
    for sz in range(len(il)+1):
        for sub in combinations(il,sz): yield set(sub)
def cf(e,att): return not any(s in e and t in e for s,t in att)
def att_of(a,att): return {s for s,t in att if t==a}
def ext_att(e,a,att): return any(s in e and t==a for s,t in att)
def defends(e,a,att): return all(ext_att(e,x,att) for x in att_of(a,att))
def admissible(e,att): return cf(e,att) and all(defends(e,a,att) for a in e)
def preferred_ext(args,att):
    adm=[s for s in all_subsets(args) if admissible(s,att)]
    return [c for c in adm if not any(c<o for o in adm)]
def stable_ext(args,att):
    out=[]
    for c in all_subsets(args):
        if not cf(c,att): continue
        if all(ext_att(c,a,att) for a in args-c): out.append(c)
    return out
def norm(exts): return [sorted(e) for e in exts]
def decisions(exts,cid="CLAIM"):
    if not exts: return {"credulous":0,"skeptical":0,"majority":0}
    cc=sum(cid in e for e in exts)
    return {"credulous":int(cc>=1),"skeptical":int(cc==len(exts)),"majority":int(cc*2>=len(exts))}
def rebuild(inter,config=BAF_CONFIG):
    nodes=dict(inter["nodes"]); rels=inter["all_audited_relations"]; sup=set(select_formal_supports(rels,config))
    sca=select_con_claim_attacks(rels,config); sp,_=select_independent_conflict_pairs(rels,config); direct=set()
    for r in sca: direct.add((r["source"],r["target"]))
    for p in sp: direct.add((p["pro"],p["con"])); direct.add((p["con"],p["pro"]))
    conn={"CLAIM"}
    for st in sup|direct: conn.add(st[0]); conn.add(st[1])
    nodes={k:v for k,v in nodes.items() if k in conn}; vids=set(nodes)
    sup={e for e in sup if e[0] in vids and e[1] in vids}; direct={e for e in direct if e[0] in vids and e[1] in vids}
    der=derive_attacks(vids,direct,sup); eff=set(map(tuple,der["effective_attacks"]))
    pref=preferred_ext(vids,eff); stab=stable_ext(vids,eff)
    return {**inter,"nodes":nodes,"supports":sorted(sup),"selected_claim_attacks":sca,
        "selected_conflict_pairs":[{"pro":p["pro"],"con":p["con"],"pair_score":p["pair_score"]} for p in sp],
        "direct_attacks":sorted(direct),"support_closure":der["support_closure"],"supported_attacks":der["supported_attacks"],
        "secondary_attacks":der["secondary_attacks"],"effective_attacks":der["effective_attacks"],
        "preferred_extensions":norm(pref),"stable_extensions":norm(stab),"preferred_decisions":decisions(pref),"stable_decisions":decisions(stab)}
def run_v3_baf_example(row,config=BAF_CONFIG):
    claim=str(row["claim"]).strip(); facts=str(row["facts_text"]).strip(); gold=int(row["gold"])
    gen=generate_and_score_arguments(claim,facts,config); sel=select_arguments(gen,config); nodes=sel["nodes"]; nodes["CLAIM"]["text"]=claim
    audit=audit_baf_relations(claim,facts,nodes,config,config["audit_batch_size"])
    inter={"example_index":int(row["example_index"]),"claim":claim,"facts_text":facts,"gold":gold,"interpreted_claim":gen["interpreted_claim"],"generated_arguments":gen,"argument_scores":sel["all_scored"],"nodes":nodes,"all_audited_relations":audit["all_audited_relations"],"model":MODEL_NAME,"config":dict(config)}
    return rebuild(inter,config)
print("✅ piece 4 ready — pipeline complete")

✅ piece 4 ready — pipeline complete


In [13]:
row = baf_df.iloc[0].copy()
row["facts_text"] = generate_facts(row["claim"])
print("CLAIM:", row["claim"])
print("FACTS:", row["facts_text"][:150])
test = run_v3_baf_example(row)
print(f"\ngold={test['gold']} pred={test['preferred_decisions']['credulous']}")
print(f"nodes={len(test['nodes'])} supports={len(test['supports'])} attacks={len(test['effective_attacks'])} extensions={len(test['preferred_extensions'])}")

CLAIM: Tim was happy that November was coming up since he'd have a lot of time off from school.
FACTS: November is a month of the year.
Students attend school.
Time off from school is called a break or holiday.
Happiness is an emotional state.

gold=0 pred=1
nodes=2 supports=1 attacks=0 extensions=1


In [10]:
CKPT = WORK_DIR / "gemma_com2sense_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))

def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (Gemma, rate-limited)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 10 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} → {len(cur)} total | running acc {acc*100:.1f}%")
        time.sleep(0.3)
    print(f"✅ {len(load_ck())}/{len(baf_df)} done")

run_batch(100)

Have 0/500. Running 100 (Gemma, rate-limited)...
  10/100 → 10 total | running acc 80.0%
  20/100 → 20 total | running acc 90.0%
  30/100 → 30 total | running acc 83.3%
  40/100 → 40 total | running acc 80.0%
  50/100 → 50 total | running acc 82.0%
  60/100 → 60 total | running acc 81.7%
  70/100 → 70 total | running acc 82.9%
  80/100 → 80 total | running acc 85.0%
  90/100 → 90 total | running acc 84.4%
  100/100 → 100 total | running acc 83.0%
✅ 100/500 done


In [11]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
pred = [r["preferred_decisions"]["credulous"] for r in results]
tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
print(f"Accuracy: {accuracy_score(gold,pred)*100:.1f}%")
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Accuracy on TRUE: {tp/(tp+fn)*100:.0f}% | on FALSE: {tn/(tn+fp)*100:.0f}%")
print(f"Predicted True: {sum(pred)} / {len(pred)} (gold True: {sum(gold)})")
# does it beat net_support?
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
ns = [net_support_pred(r) for r in results]
print(f"net_support: {accuracy_score(gold,ns)*100:.1f}%")

Accuracy: 83.0%
TN=48 FP=6 FN=11 TP=35
Accuracy on TRUE: 76% | on FALSE: 89%
Predicted True: 41 / 100 (gold True: 46)
net_support: 89.0%


The Kaggle session was restarted after the first 100 claims; the next cells reload them from a Kaggle dataset, and the run continues in batches of 100 (one claim, index 112, failed at every attempt, and the run was stopped at 497 claims, so Table 1 reports 497 claims, footnote d).

In [15]:
import glob
print("=== looking for your uploaded backup ===")
for f in glob.glob("/kaggle/input/**/*", recursive=True):
    if f.endswith(('.zip','.pkl')): print(" ", f)
print("\n=== any pkl in working ===")
for f in glob.glob("/kaggle/working/**/*.pkl", recursive=True):
    print(" ", f)

=== looking for your uploaded backup ===
  /kaggle/input/gemma-com2sense-100-backup/gemma_com2sense_100.pkl

=== any pkl in working ===


In [17]:
import pickle
upload_path = "/kaggle/input/gemma-com2sense-100-backup/gemma_com2sense_100.pkl"

data = pickle.load(open(upload_path, "rb"))
print("Loaded", len(data), "examples from backup")

# rebuild the checkpoint dict keyed by example_index
CKPT = WORK_DIR / "gemma_com2sense_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
d = {}
for i, rec in enumerate(data):
    idx = rec.get("example_index", i)
    d[idx] = rec
pickle.dump(d, open(CKPT, "wb"))
print("Checkpoint restored:", len(d), "examples")

Loaded 100 examples from backup
Checkpoint restored: 100 examples


In [20]:
CKPT = WORK_DIR / "gemma_com2sense_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))

def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (Gemma, rate-limited)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 10 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} → {len(cur)} total | running acc {acc*100:.1f}%")
        time.sleep(0.3)
    print(f"✅ {len(load_ck())}/{len(baf_df)} done")

run_batch(100)

Have 100/500. Running 100 (Gemma, rate-limited)...
  10/100 → 110 total | running acc 82.7%
error at 112: failed: Extra data: line 24 column 1 (char 682)
  20/100 → 119 total | running acc 83.2%
  30/100 → 129 total | running acc 82.9%
  40/100 → 139 total | running acc 82.0%
  50/100 → 149 total | running acc 81.9%
  60/100 → 159 total | running acc 79.2%
  70/100 → 169 total | running acc 78.7%
  80/100 → 179 total | running acc 79.9%
  90/100 → 189 total | running acc 80.4%
  100/100 → 199 total | running acc 79.9%
✅ 199/500 done


In [23]:
run_batch(100)

Have 199/500. Running 100 (Gemma, rate-limited)...
error at 112: failed: Extra data: line 24 column 1 (char 678)
  10/100 → 208 total | running acc 79.3%
  20/100 → 218 total | running acc 79.4%
  30/100 → 228 total | running acc 79.4%
  40/100 → 238 total | running acc 79.8%
  50/100 → 248 total | running acc 80.6%
  60/100 → 258 total | running acc 80.6%
  70/100 → 268 total | running acc 81.0%
  80/100 → 278 total | running acc 80.6%
  90/100 → 288 total | running acc 81.2%
  100/100 → 298 total | running acc 81.2%
✅ 298/500 done


In [26]:
run_batch(100)

Have 298/500. Running 100 (Gemma, rate-limited)...
error at 112: failed: Extra data: line 24 column 1 (char 674)
  10/100 → 307 total | running acc 81.1%
  20/100 → 317 total | running acc 81.1%
  30/100 → 327 total | running acc 80.7%
  40/100 → 337 total | running acc 81.0%
  50/100 → 347 total | running acc 80.7%
  60/100 → 357 total | running acc 80.7%
  70/100 → 367 total | running acc 80.9%
  80/100 → 377 total | running acc 81.4%
  90/100 → 387 total | running acc 81.7%
  100/100 → 397 total | running acc 81.4%
✅ 397/500 done


In [28]:
run_batch(100)

Have 397/500. Running 100 (Gemma, rate-limited)...
  10/100 → 407 total | running acc 81.3%
  20/100 → 417 total | running acc 81.5%
  30/100 → 427 total | running acc 81.5%
  40/100 → 437 total | running acc 81.5%
  50/100 → 447 total | running acc 81.7%
  60/100 → 457 total | running acc 81.8%
  70/100 → 467 total | running acc 82.0%
  80/100 → 477 total | running acc 81.8%
  90/100 → 487 total | running acc 81.9%
  100/100 → 497 total | running acc 82.1%
✅ 497/500 done


Final results on the 497 frameworks, extension statistics, and the decision rules recomputed from the saved frameworks (no API calls).

In [30]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
n = len(results)

def metrics_row(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    return {"method": name, "n": n,
        "accuracy": round(accuracy_score(gold,pred),3),
        "precision": round(precision_score(gold,pred,zero_division=0),3),
        "recall": round(recall_score(gold,pred,zero_division=0),3),
        "f1": round(f1_score(gold,pred,zero_division=0),3),
        "TN":tn,"FP":fp,"FN":fn,"TP":tp}
def no_objection_pred(r, exts_key):
    exts = r[exts_key]
    if not exts: return 0
    eff = set(map(tuple, r["effective_attacks"]))
    for ext in exts:
        es = set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff): return 1
    return 0
def net_support_pred(r):
    return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))

rows = []
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"preferred_{rule}", [r["preferred_decisions"][rule] for r in results]))
rows.append(metrics_row("preferred_no_objection", [no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"stable_{rule}", [r["stable_decisions"][rule] for r in results]))
rows.append(metrics_row("stable_no_objection", [no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(metrics_row("net_support", [net_support_pred(r) for r in results]))

table = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
print(f"=== GEMMA on COM2SENSE — FINAL, ALL rules ({n} examples) ===\n")
print(table.to_string())
print(f"\nGold balance: True={sum(gold)} False={n-sum(gold)}")
print(f"DF-QuAD 0.5 Base: 70.6% | DF-QuAD Est. Base: 78.6%")

=== GEMMA on COM2SENSE — FINAL, ALL rules (497 examples) ===

                   method    n  accuracy  precision  recall     f1   TN  FP  FN   TP
0             net_support  497     0.853      0.791   0.960  0.868  185  63  10  239
1        stable_skeptical  497     0.823      0.885   0.743  0.808  224  24  64  185
2     preferred_skeptical  497     0.823      0.885   0.743  0.808  224  24  64  185
3     preferred_credulous  497     0.821      0.864   0.763  0.810  218  30  59  190
4      preferred_majority  497     0.821      0.864   0.763  0.810  218  30  59  190
5        stable_credulous  497     0.821      0.864   0.763  0.810  218  30  59  190
6  preferred_no_objection  497     0.821      0.864   0.763  0.810  218  30  59  190
7         stable_majority  497     0.821      0.864   0.763  0.810  218  30  59  190
8     stable_no_objection  497     0.821      0.864   0.763  0.810  218  30  59  190

Gold balance: True=249 False=248
DF-QuAD 0.5 Base: 70.6% | DF-QuAD Est. Base: 78.6%


In [32]:
results = [load_ck()[k] for k in sorted(load_ck())]
n = len(results)
from collections import Counter

print(f"=== GEMMA COM2SENSE — Extension Statistics ({n} examples) ===\n")

# extensions per example
pref_counts = Counter(len(r["preferred_extensions"]) for r in results)
print("PREFERRED extensions per example:")
for k in sorted(pref_counts):
    print(f"  {k} extension(s): {pref_counts[k]:3d} examples ({pref_counts[k]/n*100:.1f}%)")
avg_ext = np.mean([len(r["preferred_extensions"]) for r in results])
max_ext = max(len(r["preferred_extensions"]) for r in results)
print(f"  → avg: {avg_ext:.2f} | max: {max_ext}")

# stable extensions
stab_counts = Counter(len(r["stable_extensions"]) for r in results)
print("\nSTABLE extensions per example:")
for k in sorted(stab_counts):
    print(f"  {k} extension(s): {stab_counts[k]:3d} examples ({stab_counts[k]/n*100:.1f}%)")
print(f"  → avg: {np.mean([len(r['stable_extensions']) for r in results]):.2f}")

# PRO kept
pro_counts = Counter(sum(1 for nid in r["nodes"] if nid.startswith("P")) for r in results)
print("\nPRO arguments kept per example:")
for k in sorted(pro_counts):
    print(f"  {k} PRO: {pro_counts[k]:3d} ({pro_counts[k]/n*100:.1f}%)")

# CON kept
con_counts = Counter(sum(1 for nid in r["nodes"] if nid.startswith("N")) for r in results)
print("\nCON arguments kept per example:")
for k in sorted(con_counts):
    print(f"  {k} CON: {con_counts[k]:3d} ({con_counts[k]/n*100:.1f}%)")

# edges
avg_sup = np.mean([len(r["supports"]) for r in results])
avg_att = np.mean([len(r["effective_attacks"]) for r in results])
print(f"\nAvg supports (edges): {avg_sup:.2f}")
print(f"Avg attacks (edges): {avg_att:.2f}")

# multi-extension
multi = sum(1 for r in results if len(r["preferred_extensions"]) > 1)
print(f"\nExamples with MULTIPLE preferred extensions: {multi} ({multi/n*100:.1f}%)")

# multi-extension accuracy by rule (the key finding)
multi_ex = [r for r in results if len(r["preferred_extensions"]) > 1]
single_ex = [r for r in results if len(r["preferred_extensions"]) == 1]
print(f"\n--- Accuracy split (the semantics finding) ---")
for name, subset in [("multi-extension", multi_ex), ("single-extension", single_ex)]:
    if subset:
        g = [r["gold"] for r in subset]
        for rule in ["credulous","skeptical","majority"]:
            p = [r["preferred_decisions"][rule] for r in subset]
            print(f"  {name} ({len(subset)}) — {rule}: {accuracy_score(g,p)*100:.1f}%")
        print()

=== GEMMA COM2SENSE — Extension Statistics (497 examples) ===

PREFERRED extensions per example:
  1 extension(s): 479 examples (96.4%)
  2 extension(s):  18 examples (3.6%)
  → avg: 1.04 | max: 2

STABLE extensions per example:
  1 extension(s): 479 examples (96.4%)
  2 extension(s):  18 examples (3.6%)
  → avg: 1.04

PRO arguments kept per example:
  0 PRO: 180 (36.2%)
  1 PRO:  49 (9.9%)
  2 PRO: 198 (39.8%)
  3 PRO:  68 (13.7%)
  4 PRO:   2 (0.4%)

CON arguments kept per example:
  0 CON: 209 (42.1%)
  1 CON:  98 (19.7%)
  2 CON: 190 (38.2%)

Avg supports (edges): 1.32
Avg attacks (edges): 1.03

Examples with MULTIPLE preferred extensions: 18 (3.6%)

--- Accuracy split (the semantics finding) ---
  multi-extension (18) — credulous: 55.6%
  multi-extension (18) — skeptical: 61.1%
  multi-extension (18) — majority: 55.6%

  single-extension (479) — credulous: 83.1%
  single-extension (479) — skeptical: 83.1%
  single-extension (479) — majority: 83.1%



In [33]:
results = [load_ck()[k] for k in sorted(load_ck())]
n = len(results)
from collections import Counter

print(f"=== GEMMA COM2SENSE — Extension counts ({n} examples) ===\n")

pref_counts = Counter(len(r["preferred_extensions"]) for r in results)
print("PREFERRED extensions per example:")
for k in sorted(pref_counts):
    print(f"  {k} extension(s): {pref_counts[k]:3d} examples ({pref_counts[k]/n*100:.1f}%)")
avg_ext = np.mean([len(r["preferred_extensions"]) for r in results])
max_ext = max(len(r["preferred_extensions"]) for r in results)
print(f"  → avg: {avg_ext:.2f} | max: {max_ext}")

stab_counts = Counter(len(r["stable_extensions"]) for r in results)
print("\nSTABLE extensions per example:")
for k in sorted(stab_counts):
    print(f"  {k} extension(s): {stab_counts[k]:3d} examples ({stab_counts[k]/n*100:.1f}%)")
print(f"  → avg: {np.mean([len(r['stable_extensions']) for r in results]):.2f}")

# conflict pairs (which drive the 2^n count)
cp_counts = Counter(len(r.get("selected_conflict_pairs",[])) for r in results)
print("\nConflict pairs per example (n → 2^n extensions):")
for k in sorted(cp_counts):
    print(f"  {k} pair(s): {cp_counts[k]:3d} ({cp_counts[k]/n*100:.1f}%) → expect {2**k} ext")

multi = sum(1 for r in results if len(r["preferred_extensions"]) > 1)
print(f"\nMultiple extensions: {multi} ({multi/n*100:.1f}%)")

=== GEMMA COM2SENSE — Extension counts (497 examples) ===

PREFERRED extensions per example:
  1 extension(s): 479 examples (96.4%)
  2 extension(s):  18 examples (3.6%)
  → avg: 1.04 | max: 2

STABLE extensions per example:
  1 extension(s): 479 examples (96.4%)
  2 extension(s):  18 examples (3.6%)
  → avg: 1.04

Conflict pairs per example (n → 2^n extensions):
  0 pair(s): 479 (96.4%) → expect 1 ext
  1 pair(s):  18 (3.6%) → expect 2 ext

Multiple extensions: 18 (3.6%)


In [4]:
import glob, pickle, zipfile
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1) find the backup: a .pkl, or a .zip to extract
pkls = [f for f in glob.glob("/kaggle/input/**/*.pkl", recursive=True) if "497" in f]
if not pkls:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if "497" in z:
            zipfile.ZipFile(z).extractall("/kaggle/working/restore")
    pkls = glob.glob("/kaggle/working/restore/**/*.pkl", recursive=True)
print("Files found:", pkls)

# 2) load the results
results = pickle.load(open(pkls[0], "rb"))
if isinstance(results, dict):                 # in case it was saved as a dict
    results = [results[k] for k in sorted(results)]
gold = [r["gold"] for r in results]
n = len(results)
print("Examples:", n, "| True:", sum(gold), "| False:", n - sum(gold))

# 3) metrics
def metrics_row(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    return {"method": name, "n": n,
        "accuracy": round(accuracy_score(gold,pred),3),
        "precision": round(precision_score(gold,pred,zero_division=0),3),
        "recall": round(recall_score(gold,pred,zero_division=0),3),
        "f1": round(f1_score(gold,pred,zero_division=0),3),
        "TN":tn,"FP":fp,"FN":fn,"TP":tp}

# CORRECTED no-objection (paper definition)
def no_objection_pred(r, exts_key):
    exts = [set(e) for e in r[exts_key]]
    if not exts or not any("CLAIM" in e for e in exts): return 0
    attackers = {s for (s, t) in map(tuple, r["effective_attacks"]) if t == "CLAIM"}
    return int(not (attackers & set().union(*exts)))

def net_support_pred(r):
    return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))

rows = []
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"preferred_{rule}", [r["preferred_decisions"][rule] for r in results]))
rows.append(metrics_row("preferred_no_objection", [no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"stable_{rule}", [r["stable_decisions"][rule] for r in results]))
rows.append(metrics_row("stable_no_objection", [no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(metrics_row("net_support", [net_support_pred(r) for r in results]))

table = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
print(f"\n=== GEMMA on COM2SENSE — corrected No-objection ({n} examples) ===\n")
print(table.to_string())

Files found: ['/kaggle/input/gemma-com2sense-497-backup/gemma_com2sense_497.pkl']
Examples: 497 | True: 249 | False: 248

=== GEMMA on COM2SENSE — corrected No-objection (497 examples) ===

                   method    n  accuracy  precision  recall     f1   TN  FP  FN   TP
0             net_support  497     0.853      0.791   0.960  0.868  185  63  10  239
1     stable_no_objection  497     0.823      0.885   0.743  0.808  224  24  64  185
2     preferred_skeptical  497     0.823      0.885   0.743  0.808  224  24  64  185
3        stable_skeptical  497     0.823      0.885   0.743  0.808  224  24  64  185
4  preferred_no_objection  497     0.823      0.885   0.743  0.808  224  24  64  185
5     preferred_credulous  497     0.821      0.864   0.763  0.810  218  30  59  190
6      preferred_majority  497     0.821      0.864   0.763  0.810  218  30  59  190
7        stable_credulous  497     0.821      0.864   0.763  0.810  218  30  59  190
8         stable_majority  497     0.821     

In [5]:
def no_obj(r, exts_key, attack_key):
    exts = [set(e) for e in r[exts_key]]
    if not exts or not any("CLAIM" in e for e in exts): return 0
    attackers = {s for (s, t) in map(tuple, r[attack_key]) if t == "CLAIM"}
    return int(not (attackers & set().union(*exts)))

direct = [no_obj(r, "preferred_extensions", "direct_attacks") for r in results]
effect = [no_obj(r, "preferred_extensions", "effective_attacks") for r in results]
skep   = [r["preferred_decisions"]["skeptical"] for r in results]

print("No-objection (direct):   ", round(accuracy_score(gold, direct) * 100, 1))
print("No-objection (effective):", round(accuracy_score(gold, effect) * 100, 1))
print("examples where direct != effective:", sum(a != b for a, b in zip(direct, effect)))
print("examples where direct != skeptical:", sum(a != b for a, b in zip(direct, skep)))

multi = [r for r in results if len(r["preferred_extensions"]) > 1]
print("\nmulti-extension examples:", len(multi))
print("of these, the claim is attacked DIRECTLY:",
      sum(any(t == "CLAIM" for (s, t) in map(tuple, r["direct_attacks"])) for r in multi))

No-objection (direct):    82.3
No-objection (effective): 82.3
examples where direct != effective: 0
examples where direct != skeptical: 0

multi-extension examples: 18
of these, the claim is attacked DIRECTLY: 18
